# Archive historique — assembler_annotations

Ce notebook est conservé pour documenter la préparation et les évaluations antérieures. Il ne doit pas être exécuté pour régénérer les jeux de test déjà validés.

In [ ]:
"""Actualise la vue consolidée et le suivi depuis la grille de travail.

Ce fichier n'est pas une vérité terrain tant que les propositions n'ont pas été
validées à partir des PDF. Usage : python Scripts/pipeline_et_validation/archive_mise_au_point/assembler_annotations.ipynb
"""

from __future__ import annotations

import csv
import json
from collections import Counter
from pathlib import Path


def trouver_racine() -> Path:
    courant = Path.cwd().resolve()
    for candidat in (courant, *courant.parents):
        if (candidat / "total_donnees").is_dir() and (candidat / "evaluation_extraction").is_dir():
            return candidat
    raise FileNotFoundError("Ouvre ce notebook depuis le dossier de travail.")

RACINE = trouver_racine()
DOSSIER = RACINE / "evaluation_extraction"
NEGATIFS_VERIFIES = {
    "CDM - 2015_12_23.pdf": "texte intégral sans rubrique de nominations",
    "CDM - 2016_01_16.pdf": "OCR français des deux pages ; aucune nomination",
    "CDM - 2017_12_30.pdf": "document de 61 pages sans rubrique de nominations",
    "CDM - 2018_12_31.pdf": "document de trois pages sans rubrique de nominations",
    "CDM - 2019_01_08.pdf": "document de trois pages sans rubrique de nominations",
}


def lire(nom: str) -> list[dict[str, str]]:
    with (DOSSIER / nom).open(newline="", encoding="utf-8-sig") as fichier:
        return list(csv.DictReader(fichier))


def ecrire(nom: str, colonnes: list[str], lignes: list[dict[str, str]]) -> None:
    with (DOSSIER / nom).open("w", newline="", encoding="utf-8-sig") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=colonnes, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(lignes)


def main() -> None:
    documents = lire("documents_echantillon.csv")
    verifies = lire("reference_annotations_validees.csv")
    documents_verifies = {ligne["document_id"] for ligne in verifies}
    lignes = lire("reference_annotations.csv")
    candidats = [ligne for ligne in lignes if ligne["statut_verification"] == "a_verifier"]
    deja_verifies = [ligne for ligne in lignes if ligne["statut_verification"] == "verifie_sur_pdf"]
    if len(deja_verifies) != len(verifies):
        raise ValueError("La grille et la référence vérifiée ne contiennent pas le même nombre de lignes vérifiées")
    ecrire("annotations_consolidees_a_valider.csv", list(lignes[0]), lignes)

    effectifs = Counter(ligne["document_id"] for ligne in candidats)
    journal = DOSSIER / "validations_documents.json"
    validations = json.loads(journal.read_text(encoding="utf-8")) if journal.exists() else {}
    suivi = []
    for document in documents:
        identifiant = document["document_id"]
        decision = validations.get(identifiant)
        if decision is not None:
            if decision["valide"]:
                statut = "annote_et_verifie" if decision["nombre_lignes"] else "aucune_nomination_verifiee"
                note = "Lecture intégrale confirmée dans l’interface."
            else:
                statut = "propositions_a_valider"
                note = "Brouillon enregistré ; document à relire intégralement."
        elif identifiant in documents_verifies:
            statut = "annote_et_verifie"
            note = "rubrique entière contrôlée dans le PDF"
        elif identifiant in NEGATIFS_VERIFIES:
            statut = "aucune_nomination_verifiee"
            note = NEGATIFS_VERIFIES[identifiant]
        else:
            statut = "propositions_a_valider"
            note = "contrôle humain des champs et de l'exhaustivité nécessaire"
        suivi.append({
            "document_id": identifiant,
            "date_nom_fichier": document["date_nom_fichier"],
            "statut": statut,
            "nombre_propositions": str(effectifs[identifiant]),
            "note": note,
        })
    ecrire("suivi_annotations.csv", list(suivi[0]), suivi)
    print(f"{len(verifies)} lignes vérifiées ; {len(candidats)} propositions ; {len(suivi)} documents")


## Exécution manuelle

Vérifier les chemins et les fichiers de sortie avant toute exécution.

In [ ]:
# main()